# Pilot: is there forgetting?
Train three stages in a row with **no replay**: **safe → helpful → quality**, one adapter the whole way.
After every stage we score the val pairs of all three datasets and save everything to Drive.

**The question:** does the model lose its safety preference once it learns helpfulness (and quality)?

* Needs a GPU: Runtime → Change runtime type → **L4**. About 2.5 hours in total.
* Keep your Mac awake: run `caffeinate -dims` in Terminal.
* **If Colab disconnects**, nothing finished is lost: set `START_STAGE` to the next stage and Run All again.

In [1]:
import os, sys

if os.path.exists("/content"):   # on Colab: get the latest code + data from GitHub
    if not os.path.exists("/content/mfr-dpo"):
        !git clone -q https://github.com/prabudhd2003/mfr-dpo.git /content/mfr-dpo
    !git -C /content/mfr-dpo pull -q
    !pip install -q peft bitsandbytes
    REPO = "/content/mfr-dpo"
else:
    REPO = ".."

sys.path.insert(0, f"{REPO}/src")
import importlib, mfr_data, mfr_dpo
importlib.reload(mfr_data)
importlib.reload(mfr_dpo)

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to a GPU")

GPU: NVIDIA L4


## Settings

In [2]:
RUN_NAME = "pilot_safe_first"
ORDER = ["safe", "helpful", "quality"]
LR, BETA, SEED = 1e-4, 0.1, 0     # LR doubled from notebook 03 so each stage learns more strongly

START_STAGE = 1                   # after a disconnect: set to the first stage that did NOT finish

# Your path to the team folder in Drive 
DRIVE_DIR = "/content/drive/MyDrive/CSCI544/mfr-dpo"
RUN_DIR = f"{DRIVE_DIR}/runs/{RUN_NAME}"

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import json
from google.colab import drive
drive.mount("/content/drive")   

os.makedirs(RUN_DIR, exist_ok=True)
with open(f"{RUN_DIR}/settings.json", "w") as f:
    json.dump({"order": ORDER, "lr": LR, "beta": BETA, "seed": SEED}, f, indent=2)

splits = mfr_data.load_splits(f"{REPO}/data")
print("saving to:", RUN_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
saving to: /content/drive/MyDrive/CSCI544/mfr-dpo/runs/pilot_safe_first


## Train the three stages
Each stage: train on that dataset's 2,000 train pairs (about 45 min), save the adapter,
then score the 200 val pairs of **all three** datasets (a few min) and save the numbers.

In [6]:
import os, sys, getpass
os.environ["HF_TOKEN"] = getpass.getpass("Paste your HF token: ")

In [7]:
import pandas as pd

results_path = f"{RUN_DIR}/results.csv"
results = pd.DataFrame()
if START_STAGE > 1 and os.path.exists(results_path):
    results = pd.read_csv(results_path)
    results = results[results["stage"] < START_STAGE]

previous = f"{RUN_DIR}/stage{START_STAGE - 1}_{ORDER[START_STAGE - 2]}" if START_STAGE > 1 else None
model, tokenizer = mfr_dpo.load_model(adapter_path=previous)
print("starting from:", previous or "a fresh adapter")

for stage in range(START_STAGE, len(ORDER) + 1):
    name = ORDER[stage - 1]
    print(f"\n===== Stage {stage}/{len(ORDER)}: training on {name} =====")
    history = mfr_dpo.train_stage(model, tokenizer, splits[name]["train"], beta=BETA, lr=LR,
                                  seed=SEED + stage, desc=f"stage {stage}/{len(ORDER)}: {name}")

    stage_dir = f"{RUN_DIR}/stage{stage}_{name}"
    model.save_pretrained(stage_dir)
    history.to_csv(f"{stage_dir}/history.csv", index=False)

    rows = []
    for eval_name in ["safe", "helpful", "quality"]:
        scores = mfr_dpo.score_pairs(model, tokenizer, splits[eval_name]["val"], beta=BETA,
                                     desc=f"scoring {eval_name} val")
        scores.to_csv(f"{stage_dir}/margins_{eval_name}_val.csv")
        rows.append({"stage": stage, "trained_on": name, "eval_set": eval_name, **mfr_dpo.summarize(scores)})

    results = pd.concat([results, pd.DataFrame(rows)], ignore_index=True)
    results.to_csv(results_path, index=False)
    print(f"saved stage {stage} to {stage_dir}")
    display(pd.DataFrame(rows).set_index("eval_set"))

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

starting from: a fresh adapter

===== Stage 1/3: training on safe =====


stage 1/3: safe:   0%|          | 0/125 [00:00<?, ?step/s]

CheckpointError: torch.utils.checkpoint: A different number of tensors was saved during the original forward and recomputation.
Number of tensors saved during forward: 99
Number of tensors saved during recomputation: 60.

Tip: To see a more detailed error message, either pass `debug=True` to
`torch.utils.checkpoint.checkpoint(...)` or wrap the code block
with `with torch.utils.checkpoint.set_checkpoint_debug_enabled(True):` to
enable checkpoint‑debug mode globally.


## Results
These cells only read the saved files, so you can run them any time later (even in a new session:
run the setup and settings cells first, then jump here).

In [ ]:
results = pd.read_csv(f"{RUN_DIR}/results.csv")
ORDER_EVAL = ["safe", "helpful", "quality"]

# Accuracy (% of val pairs where the model prefers the chosen answer) after each stage
table = results.pivot(index="eval_set", columns="stage", values="accuracy").loc[ORDER_EVAL]
table.columns = [f"after stage {s} ({ORDER[s - 1]})" for s in table.columns]
table

In [ ]:
import matplotlib.pyplot as plt

COLORS = {"safe": "#eb6834", "helpful": "#2a78d6", "quality": "#1baf7a"}   # fixed color per dataset
stages = sorted(int(s) for s in results["stage"].unique())

fig, ax = plt.subplots(figsize=(7, 3.6))
final = {}
for name in ORDER_EVAL:
    part = results[results["eval_set"] == name].sort_values("stage")
    ax.plot(part["stage"], part["accuracy"], color=COLORS[name], linewidth=2, marker="o", markersize=7, label=name)
    final[name] = part["accuracy"].iloc[-1]

# end-of-line labels, nudged apart so they never overlap
gap = 0.06 * (results["accuracy"].max() - results["accuracy"].min() + 1)
label_y, previous = {}, None
for name in sorted(final, key=final.get):
    y = final[name] if previous is None else max(final[name], previous + gap)
    label_y[name] = previous = y
for name, y in label_y.items():
    ax.annotate(f"{name} {final[name]:.0f}%", (stages[-1], final[name]), xytext=(stages[-1] + 0.08, y),
                textcoords="data", va="center", fontsize=9, color="#3d3d3a")
ax.axhline(50, color="#6b6a66", linestyle="--", linewidth=1)
ax.text(stages[0], 51, "chance", fontsize=9, color="#6b6a66", va="bottom")
ax.set_xticks(stages, [f"after {s}. {ORDER[s - 1]}" for s in stages])
ax.set_ylabel("val accuracy (%)")
ax.set_title("Preference accuracy on each val set after each stage", loc="left", fontsize=11)
ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(1.12, 1))
ax.grid(axis="y", color="#e6e5e0", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
plt.tight_layout()
plt.show()

## How much was forgotten?
For each dataset learned **before** the last stage: its score right after its own stage vs. at the end.
The per-pair columns check hypothesis H2 from the proposal: is the damage concentrated in a few pairs?

In [ ]:
last = len(ORDER)
summary = []
for k, name in enumerate(ORDER[:-1], start=1):
    right_after = results[(results["stage"] == k) & (results["eval_set"] == name)].iloc[0]
    at_end = results[(results["stage"] == last) & (results["eval_set"] == name)].iloc[0]

    m_after = pd.read_csv(f"{RUN_DIR}/stage{k}_{name}/margins_{name}_val.csv", index_col=0)["margin"]
    m_end = pd.read_csv(f"{RUN_DIR}/stage{last}_{ORDER[-1]}/margins_{name}_val.csv", index_col=0)["margin"]
    drop = (m_after - m_end).clip(lower=0)                     # how much each pair lost (0 if it didn't)
    worst_10pct = drop.sort_values(ascending=False).head(max(1, len(drop) // 10)).sum()

    summary.append({
        "dataset": name,
        "learned at stage": k,
        "accuracy right after (%)": right_after["accuracy"],
        "accuracy at the end (%)": at_end["accuracy"],
        "change (points)": round(at_end["accuracy"] - right_after["accuracy"], 1),
        "mean margin kept (%)": round(100 * at_end["mean_margin"] / right_after["mean_margin"], 0)
                                if right_after["mean_margin"] > 0 else None,
        "pairs that lost margin (%)": round(100 * (m_end < m_after).mean(), 0),
        "share of total loss from worst 10% of pairs (%)": round(100 * worst_10pct / drop.sum(), 0)
                                                           if drop.sum() > 0 else None,
    })
pd.DataFrame(summary).set_index("dataset").astype(object).T

**How to read this**
* **change (points)**: a drop of about 5 points or more on 200 val pairs is a real effect; 1–2 points is noise.
* **mean margin kept**: 100% means nothing was forgotten; 50% means half the learned preference is gone.
* **share of total loss from worst 10%**: if forgetting were spread evenly this would be about 10%.
  Much higher means a few pairs carry most of the damage, which is exactly what MFR is built to find.

If there is little forgetting, the next pilot trains each stage harder (2 epochs or a higher learning rate).